# 07 · Pooled, location-agnostic PLSR

One model maps vibration -> track quality index, with **no location feature**,
so new/relocated sensors are not out-of-distribution.

- **Features**: `X = sqrt(PSD) / speed` (set in `cfg.feature_sqrt`,
  `cfg.feature_speed_norm_power`). Applied at model time from the raw-PSD store,
  so tuning it needs no re-extraction.
- **Generalization** is measured by **leave-one-location-out** (train on other
  locations, predict the held-out one).
- **Per-location error** is reported for **monitoring only** — never a feature.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, pandas as pd
from intelliswitch import Config
DATA_ROOT = r"/Volumes/Seagate Expansion Drive/Mohammad/M Drive DTU/MATLAB Scripts/IntelliswitchScripts"
BLADE = os.path.join(DATA_ROOT, "Bladenr8252")
STORE = "feature_store"
cfg = Config()
print('features: sqrt(PSD)=%s  / speed**%.1f' % (cfg.feature_sqrt, cfg.feature_speed_norm_power))


## Load cached features, select trains, attach labels
`select_trains` applies the *selection* choices (speed > `min_speed_kmh`,
train type) — separate from extraction quality. Speed is no longer a QC
reject, so tune `cfg.min_speed_kmh` here without re-extracting.


In [ ]:
from intelliswitch.pipeline import load_feature_store, select_trains
from intelliswitch.geometry import align_campaigns, build_deg_model
from intelliswitch.geometry.deg_model import attach_labels

feat = select_trains(load_feature_store(STORE), cfg)   # speed/type selection
deg = build_deg_model(align_campaigns(BLADE, cfg), cfg)
data = attach_labels(feat, deg, cfg).dropna(subset=['y'])
print('rows after selection + labels:', len(data))
data.groupby('sensor').size()


## Train the single pooled model


In [ ]:
from intelliswitch.model import fit_pooled, evaluate, leave_one_location_out
model = fit_pooled(data, cfg)              # backend = cfg.model_kind ('plsr')
print(f'pooled {model.kind}: {model.info}  n_train={model.n_train}  train_R2={model.train_r2:.3f}')


## Overall performance + per-location monitoring
Same model everywhere; the per-location rows are a monitoring view, not
separate models.


In [ ]:
ev = evaluate(model, data, cfg)
o = ev['overall']
print(f"OVERALL  test_pts={o['n_testpoints']}  MAPE={o['mape']:.1f}%  corr={o['corr']:.2f}")
rows = [dict(sensor=n, test_pts=e['n_testpoints'], MAPE=round(e['mape'],1), corr=round(e['corr'],2))
        for n, e in ev['per_location'].items()]
pd.DataFrame(rows)


## Generalization: leave-one-location-out
Train on the other locations, predict the held-out one. If these scores are
close to the monitoring scores above, the features are location-invariant
enough to trust on new locations.


In [ ]:
rows = []
for name, e in leave_one_location_out(data, cfg).items():
    if 'mape' in e:
        rows.append(dict(held_out=name, test_pts=e['n_testpoints'],
                         MAPE=round(e['mape'],1), corr=round(e['corr'],2)))
    else:
        rows.append(dict(held_out=name, error=e.get('error')))
pd.DataFrame(rows)


## Plot: pooled prediction vs reference, per location (monitoring)


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, (name, e) in zip(axes.ravel(), ev['per_location'].items()):
    ax.plot(e['mid_days'], e['ref'],  '--o', label='reference')
    ax.plot(e['mid_days'], e['pred'], '-d',  label='pooled model')
    ax.set_title(f"{name}  MAPE={e['mape']:.1f}%  corr={e['corr']:.2f}")
    ax.set_xlabel('days before 2019-01-01'); ax.grid(True); ax.legend()
plt.tight_layout(); plt.show()


### Notes
- Tune `cfg.feature_speed_norm_power` (e.g. 0, 1, 2) and `cfg.feature_sqrt`
  here — no re-extraction needed.
- High MAPE near zero-valued references is a metric artifact; correlation is
  the stabler indicator at the test points.
